In [1]:
import torch
import numpy as np
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt

model = HookedTransformer.from_pretrained("gpt2")
model.eval()

/home/baskar/Desktop/mi-probe-suite/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint(name='hook_embed')
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint(name='hook_pos_embed')
  (blocks): ModuleList(
    (0): TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint(name='blocks.0.ln1.hook_scale')
        (hook_normalized): HookPoint(name='blocks.0.ln1.hook_normalized')
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint(name='blocks.0.ln2.hook_scale')
        (hook_normalized): HookPoint(name='blocks.0.ln2.hook_normalized')
      )
      (attn): Attention(
        (hook_k): HookPoint(name='blocks.0.attn.hook_k')
        (hook_q): HookPoint(name='blocks.0.attn.hook_q')
        (hook_v): HookPoint(name='blocks.0.attn.hook_v')
        (hook_z): HookPoint(name='blocks.0.attn.hook_z')
        (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
        (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
        (hook_result): HookPoint(name='blo

In [2]:
# A hook is a function that fires automatically during forward pass
# It receives the layer, its input, and its output

activation_store = {}

def hook_fn(module, input, output):
    # module = the layer object
    # input = tuple of inputs to this layer
    # output = what this layer produced
    activation_store["captured"] = output.detach().cpu().numpy()
    print(f"Hook fired! Output shape: {output.shape}")

# Register hook on layer 6's MLP
hook_handle = model.blocks[6].hook_mlp_out.register_forward_hook(hook_fn)

# Run forward pass
with torch.no_grad():
    logits = model("The capital of France is")

print("Captured shape:", activation_store["captured"].shape)

# ALWAYS remove hooks after use
hook_handle.remove()
print("Hook removed")

Hook fired! Output shape: torch.Size([1, 6, 768])
Captured shape: (1, 6, 768)
Hook removed


In [3]:
activation_store = {}

handles = []
for layer in range(model.cfg.n_layers):
    def make_hook(l):
        def hook_fn(module, input, output):
            activation_store[f"layer_{l}"] = output.detach().cpu().numpy()
        return hook_fn
    
    handle = model.blocks[layer].hook_resid_post.register_forward_hook(make_hook(layer))
    handles.append(handle)

with torch.no_grad():
    logits = model("The capital of France is")

# Remove all hooks
for h in handles:
    h.remove()

print("Keys captured:", list(activation_store.keys()))
print("Layer 0 shape:", activation_store["layer_0"].shape)

Keys captured: ['layer_0', 'layer_1', 'layer_2', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7', 'layer_8', 'layer_9', 'layer_10', 'layer_11']
Layer 0 shape: (1, 6, 768)


In [4]:
# TransformerLens way
_, cache = model.run_with_cache("The capital of France is")
tl_activation = cache["blocks.6.hook_resid_post"][0, -1, :].detach().numpy()

# Raw hook way
raw_store = {}
def hook_fn(module, input, output):
    raw_store["layer_6"] = output[0, -1, :].detach().numpy()

handle = model.blocks[6].hook_resid_post.register_forward_hook(hook_fn)
with torch.no_grad():
    model("The capital of France is")
handle.remove()

# Must be identical
print("Match:", np.allclose(tl_activation, raw_store["layer_6"]))

Match: True


In [5]:
import tracemalloc

tracemalloc.start()

_, cache = model.run_with_cache("The capital of France is")

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak memory: {peak / 1024 / 1024:.2f} MB")
print(f"Cache keys: {len(cache.keys())}")
print(f"Each resid_post shape: {cache['blocks.0.hook_resid_post'].shape}")

Peak memory: 0.42 MB
Cache keys: 210
Each resid_post shape: torch.Size([1, 6, 768])


In [6]:
n_prompts = 500
n_layers = model.cfg.n_layers        # 12
d_model = model.cfg.d_model          # 768
bytes_per_float = 4                   # float32

# Per prompt: storing resid_post at every layer, all token positions
avg_seq_len = 10  # approximate
memory_bytes = n_prompts * n_layers * avg_seq_len * d_model * bytes_per_float
print(f"Estimated memory: {memory_bytes / 1024 / 1024:.1f} MB")

# Now if you only store last token position:
memory_last_token = n_prompts * n_layers * 1 * d_model * bytes_per_float
print(f"Last-token only: {memory_last_token / 1024 / 1024:.1f} MB")

Estimated memory: 175.8 MB
Last-token only: 17.6 MB


In [7]:
import tracemalloc
import sys
sys.path.append("../")
from src.data.activation_extractor import extract_residual_stream, get_next_token_probs

# Without filter (old way)
tracemalloc.start()
_, cache_full = model.run_with_cache("The capital of France is")
_, peak_full = tracemalloc.get_traced_memory()
tracemalloc.stop()

# With filter (new way)
tracemalloc.start()
act = extract_residual_stream(model, "The capital of France is")
_, peak_filtered = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Full cache peak:     {peak_full / 1024 / 1024:.2f} MB")
print(f"Filtered cache peak: {peak_filtered / 1024 / 1024:.2f} MB")
print(f"Memory saved: {(peak_full - peak_filtered) / 1024 / 1024:.2f} MB")

Full cache peak:     0.42 MB
Filtered cache peak: 0.06 MB
Memory saved: 0.35 MB


In [10]:
probs = get_next_token_probs(model, "The capital of France is")
get_next_token_probs(model, "Paris is the capital of France. The capital of Germany is")
for token, prob in sorted(probs.items(), key=lambda x: -x[1]):
    print(f"  '{token}': {prob:.4f}")

  ' now': 0.0475
  ' the': 0.0374
  ' a': 0.0355
  ' home': 0.0309
  ' in': 0.0270
  ' under': 0.0257
  ' being': 0.0209
  ' set': 0.0180
  ' on': 0.0168
  ' not': 0.0149


In [ ]:
# GPT-2 tokenizes " Paris" with a leading space
paris_token = model.to_single_token(" Paris")
print("Paris token id:", paris_token)

with torch.no_grad():
    logits = model("The capital of France is")

probs = torch.softmax(logits[0, -1, :], dim=-1)
print("Prob of ' Paris':", probs[paris_token].item())

In [9]:
from src.data.activation_extractor import extract_batch

prompts = [
    "The capital of France is",
    "The capital of Germany is",
    "The capital of Japan is",
    "I like to eat pizza with",
]

activations = extract_batch(model, prompts)
print("Shape:", activations.shape)  # (4, 12, 768)
assert activations.shape == (4, 12, 768), "Shape mismatch!"
print("All checks passed ✅")

100%|████████████████████████████████████████████| 4/4 [00:00<00:00,  9.22it/s]

Shape: (4, 12, 768)
All checks passed ✅
